In [ ]:
import os, sys, h5py
import time

import numpy as np
import skimage.transform as transform
import matplotlib.pyplot as plt
import matplotlib.colors

import seaborn as sns
sns.set_theme()

from IPython.display import clear_output

if os.path.exists('/home/ayyerkar/.local/dragonfly/utils/py_src/'):
    sys.path.append('/home/ayyerkar/.local/dragonfly/utils/py_src/')
    print('Appended Dragonfly!')
    import detector
    import reademc
    import writeemc
else:
    print('Dragonfly not found, please install Dragonfly!')

dsf = 4
det_file = f'emc/make_detector/det_agipd_9_kev_ds_{dsf}.h5'
#det_file = f'emc/make_detector/det_agipd_9_kev_ds_{dsf}_no_mask.h5' # no_mask appended to end if no detector mask
det = detector.Detector(det_fname=det_file, mask_flag=True, keep_mask_1=True)

writeEMC = False
writeEMCWaterOnly = False

<h2> Writing EMC / EMCWaterOnly files </h2>

For the AGIPD simulations, the edge pixel is technically "off the detector" for the bottom horizontal edge. This only really affects the edge resolution number as defined for PRTF/FSC. 
The solution is to crop the patterns to 90 by 90 for this data. The PRTF will never really intersect the 1/e threshold at the edge resolution for all the experimental conditions I am looking
at, so this is not really that big of an issue. And this would only affect the plot when the resolution limits are displayed. 

In [ ]:
if writeEMC:
    pat_num = '1000k' # 100k or 1000k

    #type_ext = ['_masked','_with_water_masked']
    type_ext = [''] # unmasked data
    
    min_run, max_run = 0, 500 # "0, 500" for 1000k or "0, 50" for 100k
    for ext in type_ext:
        sys.stderr.write(f'Converting {ext} now...\n')
        for r in range(min_run, max_run):
            sys.stderr.write(f'Writing emc file for run {r+1}/{max_run}...\n')
            simulation_run = f'sims_protein_water/run_{r}_protein_in_water_{pat_num}_pats_dsf_{dsf}x/poisson_prot{ext}.npy'
            simulation_number = simulation_run.split(sep='/')[1]
            patterns = np.load(simulation_run)
            if patterns.dtype == 'float64':
                patterns = patterns.astype(np.int64)
            out_fname = f'sparse_frames_protein_water/'+f'{simulation_number}'+f'{ext}.emc'
            wemc = writeemc.EMCWriter(out_fname, patterns.shape[1]*patterns.shape[2], hdf5=False)
            for f in range(patterns.shape[0]):
                frame = patterns[f].ravel()
                frame[frame < 0.] = 0.
                wemc.write_frame(frame)
                sys.stderr.write('\r%d/%d'%(f+1, patterns.shape[0]))
            sys.stderr.write('\n')
            wemc.finish_write()
            clear_output(wait=False)
        sys.stderr.write(f'Finished writing {max_run-min_run} file(s)...\n')

if writeEMCWaterOnly:
    pat_num = '1000k' # 100k or 1000k
    
    ext = 'poisson_water_only_masked'
    #ext = 'poisson_water_only' # unmasked data
    
    min_run, max_run = 0, 5 # "0, 5" for 100k or "0, 50" for 1000k 
    for r in range(min_run, max_run):
        sys.stderr.write(f'Writing emc background file for run {r+1}/{max_run}...\n')
        simulation_run = f'sims_water_only/run_{r}_water_{pat_num}_pats_dsf_{dsf}x/{ext}.npy'
        simulation_number = simulation_run.split(sep='/')[1]
        patterns = np.load(simulation_run)
        if patterns.dtype == 'float64':
            patterns = patterns.astype(np.int64)
        out_fname = f'sparse_frames_water/'+f'{simulation_number}'+'_'+f'{ext}.emc'
        wemc = writeemc.EMCWriter(out_fname, patterns.shape[1]*patterns.shape[2], hdf5=False)
        for f in range(patterns.shape[0]): 
            frame = patterns[f].ravel()
            frame[frame < 0.] = 0.
            wemc.write_frame(frame)
            sys.stderr.write('\r%d/%d'%(f+1, patterns.shape[0]))
        sys.stderr.write('\n')
        wemc.finish_write()
        clear_output(wait=False)
    sys.stderr.write(f'Finished writing {max_run-min_run} file(s)...\n')

<h2> Inspecting written EMC files with no water diffraction </h2> 

In [ ]:
run_nr = 0
npats = '1000k'
ext = f'dsf_{dsf}x_masked'

emc_1 = reademc.EMCReader([f'sparse_frames_protein_water/run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det])
emc_2 = reademc.EMCReader([f'sparse_frames_protein_water/run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
emc_3 = reademc.EMCReader([f'sparse_frames_protein_water/run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
rng = np.random.default_rng()
r_pat1 = rng.integers(0,2000)
r_pat2 = rng.integers(0,2000)
r_pat3 = rng.integers(0,2000)

emc_frame1 = emc_1.get_frame(r_pat1,sym=False)
emc_frame2 = emc_2.get_frame(r_pat2,sym=False)
emc_frame3 = emc_3.get_frame(r_pat3,sym=False)

fig_handle = plt.figure(constrained_layout = True, dpi = 200)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
min_v = 0
max_v = 2
cm = plt.get_cmap('viridis', max_v+1)
#cm = 'viridis'

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(emc_frame1,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_0.set_title(f'({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_xticks([])
ax_0.set_yticks([])
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.43,orientation='vertical')
c_bar_0.set_ticks(np.arange(min_v, max_v+1))

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(emc_frame2,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_1.set_title(f'({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_xticks([])
ax_1.set_yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(emc_frame3,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_2.set_title(f'({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_xticks([])
ax_2.set_yticks([]);

<h2> Inspecting written EMC files with water diffraction </h2> 

In [ ]:
run_nr = 0
npats = '100k'
ext = f'dsf_{dsf}x_with_water_masked'

emc_1 = reademc.EMCReader([f'sparse_frames_protein_water/run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det])
emc_2 = reademc.EMCReader([f'sparse_frames_protein_water/run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
emc_3 = reademc.EMCReader([f'sparse_frames_protein_water/run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 

rng = np.random.default_rng()
r_pat1 = rng.integers(0,2000)
r_pat2 = rng.integers(0,2000)
r_pat3 = rng.integers(0,2000)

emc_frame1 = emc_1.get_frame(r_pat1,sym=False)
emc_frame2 = emc_2.get_frame(r_pat2,sym=False)
emc_frame3 = emc_3.get_frame(r_pat3,sym=False)

fig_handle = plt.figure(constrained_layout = True, dpi = 200)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
min_v = 0
max_v = 3
cm = plt.get_cmap('viridis', max_v+1)

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(emc_frame1,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_0.set_title(f'run {run_nr} - {r_pat1} ({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_title(f'({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_xticks([])
ax_0.set_yticks([])
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.43,orientation='vertical')
c_bar_0.set_ticks(np.arange(min_v, max_v+1))

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(emc_frame2,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_1.set_title(f'run {run_nr} - {r_pat2} ({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_title(f'({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_xticks([])
ax_1.set_yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(emc_frame3,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_2.set_title(f'run {run_nr} - {r_pat3} ({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_title(f'({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_xticks([])
ax_2.set_yticks([]);

<h2> Inspecting written EMC files with only water diffraction </h2> 

In [ ]:
run_nr = 0
npats = '100k'
ext = f'dsf_{dsf}x_poisson_water_only_masked'

emc_1 = reademc.EMCReader([f'sparse_frames_water/run_{run_nr}_water_{npats}_pats_{ext}.emc'], geom_list=[det])
emc_2 = reademc.EMCReader([f'sparse_frames_water/run_{run_nr}_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
emc_3 = reademc.EMCReader([f'sparse_frames_water/run_{run_nr}_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
_
rng = np.random.default_rng()
r_pat1 = rng.integers(0,10000)
r_pat2 = rng.integers(0,10000)
r_pat3 = rng.integers(0,10000)

emc_frame1 = emc_1.get_frame(r_pat1,sym=False)
emc_frame2 = emc_2.get_frame(r_pat2,sym=False)
emc_frame3 = emc_3.get_frame(r_pat3,sym=False)

fig_handle = plt.figure(constrained_layout = True, dpi = 200)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
min_v = 0
max_v = 2
cm = plt.get_cmap('viridis', max_v+1)

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(emc_frame1,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_0.set_title(f'run {run_nr} - {r_pat1} ({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_title(f'({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_xticks([])
ax_0.set_yticks([])
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.43,orientation='vertical')
c_bar_0.set_ticks(np.arange(min_v, max_v+1))

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(emc_frame2,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_1.set_title(f'run {run_nr} - {r_pat2} ({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_title(f'({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_xticks([])
ax_1.set_yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(emc_frame3,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_2.set_title(f'run {run_nr} - {r_pat3} ({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_title(f'({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_xticks([])
ax_2.set_yticks([]);

<h2> Checking beta parameter schedule and rotational sampling </h2> 
Note that Dragonfly determines the beta schedule by modulo dividing the number of iterations by the jump schedule.
This means that if the remainder is not equal to 0, there will be a few iterations with a higher beta than as set
by the schedule. However, for beta equal to 1 - so no annealing. 

For the rotational sampling, running the equation to get the number rotations as a function of n and the difference between each rotation use:
$$ n_{rotations}=50n^3+10n $$
$$ \delta_{rot}=\frac{0.944}{n} $$
and we get for the first equation: $$60, 420, 1380, 3240, 6300, 10860, 17220, 25680, 36540, 50100, 66660, 86520, 109980$$ 
For the second equation we get: $$0.944, 0.472, 0.31466667, 0.236, 0.1888, 0.15733333, 0.13485714, 0.118, 0.10488889, 0.0944, 0.08581818, 0.07866667, 0.07261538$$

In [ ]:
beta = 0.01

beta_jump, beta_schedule = 1.4141, 8
max_iter = 120
b_list = [beta]
max_iter = max_iter//beta_schedule - 1

for i in range(max_iter):
    beta*=beta_jump
    b_list.append(beta)
    
plotBeta = True
if plotBeta:
    plt.figure(dpi=120)
    plt.plot(b_list,'o--')
    plt.xticks(ticks=np.arange(0,max_iter+1,3))
    plt.yticks(ticks=[b_list[0],b_list[-1]])
    plt.xlabel('jump iteration', weight='bold')
    plt.ylabel('beta', weight='bold');
    print(f'Beta schedule: {b_list}')
else:
    num_div = 3
    num_rot = 50*(num_div)**3+10*num_div
    print(f'Rotational samples: {num_rot}')